# In-shield coil design

Designs a coil on a cylindrical former inside a closed magnetic shield, using the Nottingham `fields_in_shields` modules unchanged. Set parameters in cell 2, run top to bottom. Run the last cell once to confirm this notebook reproduces their script.

**1. Setup.** Their `modules/` folder is imported as a library; nothing in `fields_in_shields-master` is edited.

In [ ]:
%matplotlib widget
import os, sys, json, subprocess, tempfile, glob
import numpy as np
import matplotlib.pyplot as plt

plt.rc('font', family='serif')
plt.rc('xtick', labelsize=14)
plt.rc('ytick', labelsize=14)
plt.rc('axes', titlesize=16, labelsize=16)
plt.rc('legend', fontsize=14)
plt.rc('figure', titlesize=18)

FIS = os.path.abspath('fields_in_shields-master')
sys.path.insert(0, FIS)
from modules import targetpoints as tpts, sphericalharmonics as sh, componentfunctions as cf, biotsavart

**2. Parameters.** Units: meters, tesla, amps. Shield is centered at the origin, axis along z.

In [ ]:
p = dict(
    name='By_stock',        # run folder name under runs/

    # shield
    a=0.25,                 # inner radius
    L=1.0,                  # inner length, cap to cap

    # former
    rho_c=0.2,              # radius
    z_top=0.45 + 1e-7,      # winding top; z_bot = -z_top
    # The +1e-7 avoids a bug in their code: if coil length / shield length is a simple
    # fraction (0.9 here), some terms are silently dropped and the predicted field is wrong.

    # target field: function name from sphericalharmonics.py
    # b1m1 = uniform By, b1p1 = uniform Bx, b10 = uniform Bz, b20 = dBz/dz, b2m1 = dBy/dz
    target='b1m1',
    scale=1.0,              # T (T/m for gradients). Fit is linear; real current is scaled in cell 5.
    plot_comp=1,            # component to plot: 0 = x, 1 = y, 2 = z

    # fit region: cylinder of points (rho, phi, z ranges and counts)
    rho_tp=(0.01, 0.12),    # 0.6 x coil radius
    phi_tp=(0, 1.5*np.pi),  # stops short of 2 pi so no point repeats
    z_tp=(-0.27, 0.27),     # 0.6 x coil half-length
    n_tp=(4, 4, 5),

    # fit settings
    N=50,                   # axial modes
    M=1,                    # highest azimuthal order (1 for uniform fields and d/dz gradients)
    p_max=25,               # end-cap image terms kept
    beta=5e-15 * (1e-3 / 1.68e-8),  # power penalty: larger = simpler, weaker coil
    res=1.68e-8, t=1e-3,    # copper resistivity, sheet thickness (only scale the penalty)

    # wires
    n_wires=18,             # contour levels to wind
    n_phi_grid=200, n_z_grid=500,   # stream function sampling before contouring
    B_want=100e-12,         # field you actually want (T); sets current per wire
)

**3. Pipeline.** Thin wrappers that call their functions in the same order as `generatewires_cylindrical.py`.

In [ ]:
def geom(p):
    return np.array([p['rho_c']]), np.array([p['z_top']]), np.array([-p['z_top']])

def containers(p, pts_cyl):
    # field of every current mode at every point, shield included (the slow step)
    r, z1, z2 = geom(p)
    return cf.integral_containers(pts_cyl, r, z1, z2, [p['N'], p['M'], p['p_max'], p['a'], p['L']])

def fit(p):
    r, z1, z2 = geom(p)
    tp = tpts.cylinder_grid_cylindrical([*p['rho_tp'], *p['phi_tp'], *p['z_tp']], list(p['n_tp']))
    target = sh.b_sum(getattr(sh, p['target'])(scale=p['scale']))
    B_des = tpts.fieldcarttopol(sh.b_sphericalharmonics(tpts.poltocart(tp), target), tp)
    c0, c, d = containers(p, tp)
    P0 = cf.power_matrix_0(r, z1, z2, [p['N'], p['M'], p['res'], p['t']])
    P = cf.power_matrix(r, z1, z2, [p['N'], p['M'], p['res'], p['t']])
    return dict(target=target,
                W0=cf.fourier_calculation_0(c0, B_des, P0, p['beta']),
                W=cf.fourier_calculation(c, B_des, P, p['beta']),
                Q=cf.fourier_calculation(d, B_des, P, p['beta']))

def lines(p):
    # same check lines as their script: along x through the center, and along the axis over the coil length
    x = tpts.cubic_grid_cylindrical([2*p['rho_c'], 1e-6, 1e-6], [101, 1, 1], np.zeros(3))
    z = tpts.cubic_grid_cylindrical([1e-6, 1e-6, 2*p['z_top']], [1, 1, 101], np.zeros(3))
    return x, z

def field(p, d, pts_cyl):
    # predicted field (Cartesian) of the ideal current sheet inside the shield, and the target
    c0, c, q = containers(p, pts_cyl)
    B = tpts.fieldpoltocart(cf.b_calculation(pts_cyl, c0, d['W0'], c, d['W'], q, d['Q']), pts_cyl)
    return B, sh.b_sphericalharmonics(tpts.poltocart(pts_cyl), d['target'])

def wires(p, d):
    # contour the stream function into wire loops; every loop carries the same current
    r, z1, z2 = geom(p)
    nphi, nz = p['n_phi_grid'], p['n_z_grid']
    grid = tpts.wirepoints_cylindrical([p['rho_c'], p['z_top'], -p['z_top']], [nphi, nz])
    sf = np.fliplr(cf.streamfunction_calculation(grid, d['W0'], d['W'], d['Q'], z1[0], z2[0],
                                                 [p['N'], p['M']]).reshape(nphi, nz))
    step = (sf.max() - sf.min()) / p['n_wires']          # current per wire (A) at the target scale
    levels = sf.min() + (np.arange(1, p['n_wires'] + 1) - 0.5) * step
    X, Y = np.meshgrid(np.arange(nz), np.arange(nphi))
    segs = plt.contour(X, Y, sf, levels=levels).allsegs
    plt.close()
    cart, cyl = tpts.poltocartcontour(segs, [p['rho_c'], p['z_top'], -p['z_top']], [nphi, nz])
    return cart, cyl, step

**4. Fit and check the field.** Prediction assumes an ideal closed shield and a smooth current sheet.

In [ ]:
d = fit(p)
xl, zl = lines(p)
Bx_line, Tx_line = field(p, d, xl)
Bz_line, Tz_line = field(p, d, zl)
k, comp = p['plot_comp'], 'xyz'[p['plot_comp']]
ref = np.max(np.abs(Tz_line[:, k]))

plt.figure()
plt.plot(zl[:, 2], Tz_line[:, k], 'k', label='target')
plt.plot(zl[:, 2], Bz_line[:, k], 'r--', label='predicted')
for zz in p['z_tp']:
    plt.axvline(zz, color='grey', ls=':')
plt.xlabel('z (m)'); plt.ylabel(f'B{comp} (T)'); plt.title('Along the axis'); plt.legend()

plt.figure()
plt.plot(zl[:, 2], 100 * (Bz_line[:, k] - Tz_line[:, k]) / ref)
for zz in p['z_tp']:
    plt.axvline(zz, color='grey', ls=':')
plt.xlabel('z (m)'); plt.ylabel('error (% of target)'); plt.title(f'B{comp} error along the axis')

plt.figure()
plt.plot(xl[:, 0], Tx_line[:, k], 'k', label='target')
plt.plot(xl[:, 0], Bx_line[:, k], 'r--', label='predicted')
for xx in (-p['rho_tp'][1], p['rho_tp'][1]):
    plt.axvline(xx, color='grey', ls=':')
plt.xlabel('x (m)'); plt.ylabel(f'B{comp} (T)'); plt.title('Along x through the center'); plt.legend()

inside = (zl[:, 2] >= p['z_tp'][0]) & (zl[:, 2] <= p['z_tp'][1])
print(f'max error inside fit region along axis: {100*np.max(np.abs(Bz_line[inside, k]-Tz_line[inside, k]))/ref:.3f} %')

**5. Wires.** Unrolled former: angle across, height up. Solid and dashed carry current in opposite senses.

In [ ]:
w_cart, w_cyl, step = wires(p, d)
I_wire = step * p['B_want'] / p['scale']
styles = biotsavart.linestyle_cylindrical(w_cyl)

plt.figure()
for w, s in zip(w_cyl, styles):
    plt.plot(w[:, 1] / np.pi, w[:, 2], 'k', ls=s, lw=1)
plt.xlim(0, 2); plt.xlabel('phi / pi'); plt.ylabel('z (m)'); plt.title('Wire pattern')

# loops crossing phi = 0 come out as two half files; count them so it is not a surprise
n_open = sum(not np.allclose(w[0], w[-1], atol=1e-6) for w in w_cart)
print(f'{len(w_cart)} wire files, {n_open} of them cut at phi = 0 (join pairs before winding)')
print(f'current per wire for {p["B_want"]:.3g} T: {I_wire:.3g} A')

**6. Save.** Writes `runs/<name>/`: parameters, mode amplitudes, one file per wire (x, y, z in m, point order = current direction).

In [ ]:
out = os.path.join('runs', p['name'])
os.makedirs(os.path.join(out, 'wires'), exist_ok=True)
for f in glob.glob(os.path.join(out, 'wires', '*.txt')):
    os.remove(f)
with open(os.path.join(out, 'params.json'), 'w') as fh:
    json.dump({**p, 'I_wire_A': I_wire}, fh, indent=1)
for key in ('W0', 'W', 'Q'):
    np.savetxt(os.path.join(out, f'{key}.csv'), d[key], delimiter=',')
for i, w in enumerate(w_cart):
    np.savetxt(os.path.join(out, 'wires', f'{i:03d}.txt'), w, delimiter=',')
print('saved to', out)

**7. Check against their script (run once).** Runs their unmodified `generatewires_cylindrical.py` in a temporary folder with its stock settings, runs this notebook's pipeline with the same settings, and compares. Everything should agree to rounding (~1e-12). Takes about 30 s.

In [ ]:
stock = {**p, 'name': 'stock_check', 'target': 'b1p1', 'scale': 1.0, 'z_top': 0.45,   # their exact settings, no nudge
         'rho_tp': (0.01, 0.12), 'phi_tp': (0, 1.5*np.pi), 'z_tp': (0.6*-0.45, 0.6*0.45), 'n_tp': (4, 4, 5),
         'N': 50, 'M': 1, 'p_max': 25, 'beta': 5e-15 * (1e-3 / 1.68e-8), 'n_wires': 18}

with tempfile.TemporaryDirectory() as tmp:
    env = {**os.environ, 'MPLBACKEND': 'Agg', 'PYTHONPATH': FIS}
    run = subprocess.run([sys.executable, os.path.join(FIS, 'generatewires_cylindrical.py')],
                         cwd=tmp, env=env, capture_output=True, text=True)
    assert run.returncode == 0, run.stderr[-2000:]
    theirs = os.path.join(tmp, 'simulations', 'Bx')
    ref = {k: np.loadtxt(os.path.join(theirs, f'{k}.csv'), delimiter=',') for k in ('W0', 'W', 'Q', 'B_analytic_mesz')}
    ref_wires = [np.loadtxt(f, delimiter=',') for f in
                 sorted(glob.glob(os.path.join(theirs, 'contours_levels_18', '*.txt')),
                        key=lambda f: int(os.path.basename(f).split('-')[0]))]

ds = fit(stock)
B_mine, _ = field(stock, ds, lines(stock)[1])
w_mine = wires(stock, ds)[0]

def rel(a, b):
    a, b = np.ravel(a), np.ravel(b)
    return np.max(np.abs(a - b)) / np.max(np.abs(b))

checks = {'mode amplitudes W0': rel(ds['W0'], ref['W0']),
          'mode amplitudes W': rel(ds['W'], ref['W']),
          'mode amplitudes Q': rel(ds['Q'], ref['Q']),
          'field along axis': rel(B_mine, ref['B_analytic_mesz'])}
same_count = len(w_mine) == len(ref_wires)
checks['wire coordinates'] = max(rel(a, b) for a, b in zip(w_mine, ref_wires)) if same_count else np.inf
for name, v in checks.items():
    print(f'{name:22s} max relative difference {v:.1e}   {"PASS" if v < 1e-9 else "FAIL"}')
print(f'wire count: mine {len(w_mine)}, theirs {len(ref_wires)}   {"PASS" if same_count else "FAIL"}')